# 03 - BERT Model Comparison

Detaillierter Vergleich der drei BERT-Modelle (mBERT, GBERT, HateBERT).

**Inhalte:**
- Cross-Validation Ergebnisse analysieren
- Per-Class Performance vergleichen
- Statistische Signifikanztests
- Learning Curves aus Experiment 3
- Preprocessing Ablation aus Experiment 4

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.config import METRICS_DIR, PLOTS_DIR, LABEL_NAMES

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('Setup complete!')

Setup complete!


## 1. CV-Ergebnisse laden

In [2]:
# Lade alle BERT CV-Ergebnisse
models = ['GBERT', 'mBERT', 'HateBERT']
results = {}

for model_name in models:
    result_path = METRICS_DIR / f'{model_name}_cv_results.json'
    if result_path.exists():
        with open(result_path, 'r') as f:
            results[model_name] = json.load(f)
        print(f'✅ {model_name}: Geladen')
    else:
        print(f'⚠️  {model_name}: Nicht gefunden')

# Als DataFrame
df_summary = pd.DataFrame({
    model: {
        'F1 (Macro)': f"{data['f1_macro_mean']:.4f} ± {data['f1_macro_std']:.4f}",
        'Accuracy': f"{data['accuracy_mean']:.4f} ± {data['accuracy_std']:.4f}",
        'F1 OTHER': f"{data.get('f1_OTHER_mean', 0):.4f}",
        'F1 OFFENSE': f"{data.get('f1_OFFENSE_mean', 0):.4f}",
    }
    for model, data in results.items()
}).T

print('\n=== BERT-Modelle Zusammenfassung ===')
print(df_summary)

✅ GBERT: Geladen
✅ mBERT: Geladen
✅ HateBERT: Geladen

=== BERT-Modelle Zusammenfassung ===
               F1 (Macro)         Accuracy F1 OTHER F1 OFFENSE
GBERT     0.7923 ± 0.0120  0.8210 ± 0.0090   0.8694     0.7153
mBERT     0.7684 ± 0.0155  0.7965 ± 0.0103   0.8489     0.6880
HateBERT  0.6724 ± 0.0058  0.7123 ± 0.0102   0.7857     0.5590


## 2. Statistische Signifikanztests

In [3]:
# Paired t-test zwischen Modellen
print('\n=== Paarweise t-Tests (F1-Scores über 5 Folds) ===\n')

model_pairs = [
    ('GBERT', 'mBERT'),
    ('GBERT', 'HateBERT'),
    ('mBERT', 'HateBERT'),
]

for model_a, model_b in model_pairs:
    if model_a in results and model_b in results:
        f1_a = results[model_a].get('f1_macro_values', [])
        f1_b = results[model_b].get('f1_macro_values', [])
        
        if len(f1_a) == len(f1_b) and len(f1_a) > 0:
            t_stat, p_value = stats.ttest_rel(f1_a, f1_b)
            mean_diff = np.mean(f1_a) - np.mean(f1_b)
            
            sig_marker = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'n.s.'
            
            print(f'{model_a} vs. {model_b}:')
            print(f'  Differenz: {mean_diff:+.4f}')
            print(f'  t-Statistik: {t_stat:.3f}')
            print(f'  p-Wert: {p_value:.4f} {sig_marker}')
            print()


=== Paarweise t-Tests (F1-Scores über 5 Folds) ===

GBERT vs. mBERT:
  Differenz: +0.0239
  t-Statistik: 2.818
  p-Wert: 0.0479 *

GBERT vs. HateBERT:
  Differenz: +0.1200
  t-Statistik: 20.380
  p-Wert: 0.0000 ***

mBERT vs. HateBERT:
  Differenz: +0.0961
  t-Statistik: 17.421
  p-Wert: 0.0001 ***



## 3. Per-Class Performance

In [4]:
# 3. Per-Class Performance
# Visualisierung ausgelagert in scripts/generate_all_plots.py
# Siehe: results/plots/per_class_metrics.png

print('\n✅ Plot "per_class_metrics.png" wird durch scripts/generate_all_plots.py erstellt.')

# Tabellarische Übersicht F1 pro Modell & Klasse anzeigen
for model in models:
    if model in results:
        label0_f1 = results[model].get(f'f1_{LABEL_NAMES[0]}_mean', 0)
        label1_f1 = results[model].get(f'f1_{LABEL_NAMES[1]}_mean', 0)
        print(f"{model}:\n  {LABEL_NAMES[0]}: {label0_f1:.4f}\n  {LABEL_NAMES[1]}: {label1_f1:.4f}")



✅ Plot "per_class_metrics.png" wird durch scripts/generate_all_plots.py erstellt.
GBERT:
  OTHER: 0.8694
  OFFENSE: 0.7153
mBERT:
  OTHER: 0.8489
  OFFENSE: 0.6880
HateBERT:
  OTHER: 0.7857
  OFFENSE: 0.5590


## 4. Learning Curve (Experiment 3)

In [5]:
# 4. Learning Curve (Experiment 3)
# Visualisierung ausgelagert in scripts/generate_all_plots.py
# Siehe: results/plots/learning_curve.png

print('\n✅ Plot "learning_curve.png" wird durch scripts/generate_all_plots.py erstellt.')

# Deskriptiv:
data_size_path = METRICS_DIR / 'data_size_variation_results.json'

if data_size_path.exists():
    with open(data_size_path, 'r') as f:
        data_size_results = json.load(f)
    print('\n=== Data Efficiency Stats ===')
    res_dict = data_size_results.get("results", data_size_results)
    for size_key in sorted(res_dict.keys(), key=lambda x: float(x.strip('%'))):
        data = res_dict[size_key]
        print(f'{size_key} Daten: F1={data["f1_macro_mean"]:.4f}, Acc={data["accuracy_mean"]:.4f}')
else:
    print('⚠️  Experiment 3 Daten nicht gefunden.')



✅ Plot "learning_curve.png" wird durch scripts/generate_all_plots.py erstellt.

=== Data Efficiency Stats ===
10% Daten: F1=0.5214, Acc=0.5845
25% Daten: F1=0.7097, Acc=0.7651
50% Daten: F1=0.7880, Acc=0.8144
75% Daten: F1=0.7914, Acc=0.8208
100% Daten: F1=0.8097, Acc=0.8334


## 5. Preprocessing Ablation (Experiment 4)

In [6]:
# 5. Preprocessing Ablation (Experiment 4)
# Visualisierung ausgelagert in scripts/generate_all_plots.py
# Siehe: results/plots/preprocessing_ablation.png

print('\n✅ Plot "preprocessing_ablation.png" wird durch scripts/generate_all_plots.py erstellt.')

# Deskriptiv:
preprocessing_path = METRICS_DIR / 'preprocessing_ablation_results.json'

if preprocessing_path.exists():
    with open(preprocessing_path, 'r') as f:
        preprocessing_data = json.load(f)
    print('\n=== Preprocessing Stats (Top 5) ===')
    results_dict = preprocessing_data.get('results', {})
    sorted_res = sorted(results_dict.items(), key=lambda x: x[1]['f1_macro_mean'], reverse=True)
    for k, v in sorted_res[:5]:
        print(f"{k}: F1={v['f1_macro_mean']:.4f}")
else:
    print('⚠️  Experiment 4 Daten nicht gefunden.')



✅ Plot "preprocessing_ablation.png" wird durch scripts/generate_all_plots.py erstellt.

=== Preprocessing Stats (Top 5) ===
full_preprocessing: F1=0.8097
remove_urls: F1=0.8053
normalize_usernames: F1=0.8050
original: F1=0.8036
remove_emojis: F1=0.8031


## 6. Zusammenfassung für Paper

In [7]:
print('\n' + '='*70)
print('ZUSAMMENFASSUNG FÜR HAUSARBEIT')
print('='*70)

# Bestes Modell
best_model = max(results.items(), key=lambda x: x[1]['f1_macro_mean'])
print(f'\n✅ Bestes Modell: {best_model[0]}')
print(f'   F1 (Macro):  {best_model[1]["f1_macro_mean"]:.4f} ± {best_model[1]["f1_macro_std"]:.4f}')
print(f'   Accuracy:    {best_model[1]["accuracy_mean"]:.4f} ± {best_model[1]["accuracy_std"]:.4f}')

# Modell-Ranking
print('\n📊 Modell-Ranking (nach F1):')
for i, (model, data) in enumerate(sorted(results.items(), key=lambda x: x[1]['f1_macro_mean'], reverse=True), 1):
    print(f'   {i}. {model}: {data["f1_macro_mean"]:.4f}')

# Key Findings
print('\n🔍 Key Findings:')
if 'GBERT' in results and 'mBERT' in results:
    gbert_f1 = results['GBERT']['f1_macro_mean']
    mbert_f1 = results['mBERT']['f1_macro_mean']
    diff = gbert_f1 - mbert_f1
    print(f'   • GBERT übertrifft mBERT um {diff:.4f} F1-Punkte ({diff/mbert_f1*100:.1f}%)')

if 'GBERT' in results and 'HateBERT' in results:
    gbert_f1 = results['GBERT']['f1_macro_mean']
    hatebert_f1 = results['HateBERT']['f1_macro_mean']
    diff = gbert_f1 - hatebert_f1
    print(f'   • GBERT übertrifft HateBERT um {diff:.4f} F1-Punkte ({diff/hatebert_f1*100:.1f}%)')

print(f'   • Language-specific model (GBERT) schlägt multilingual (mBERT)')
print(f'   • Domain-specific falscher Sprache (HateBERT) schlechter als generisches GBERT')


ZUSAMMENFASSUNG FÜR HAUSARBEIT

✅ Bestes Modell: GBERT
   F1 (Macro):  0.7923 ± 0.0120
   Accuracy:    0.8210 ± 0.0090

📊 Modell-Ranking (nach F1):
   1. GBERT: 0.7923
   2. mBERT: 0.7684
   3. HateBERT: 0.6724

🔍 Key Findings:
   • GBERT übertrifft mBERT um 0.0239 F1-Punkte (3.1%)
   • GBERT übertrifft HateBERT um 0.1200 F1-Punkte (17.8%)
   • Language-specific model (GBERT) schlägt multilingual (mBERT)
   • Domain-specific falscher Sprache (HateBERT) schlechter als generisches GBERT
